# KDE on LoG punctum centroids for dendrite masks

This notebook tests a point-cloud dendrite detector on a small patch subset. Instead of smoothing the raw structural intensity, it first detects LoG puncta on the structural channel and then smooths only the resulting centroid field. The density map is therefore driven by scale-consistent puncta, not by pixel-wise noise.

## Why centroid KDE instead of intensity KDE?
1. `detect_blobs_log` finds scale-normalised LoG maxima on the structural channel and returns `(row, col, sigma)` centroids.
2. Each centroid becomes a unit impulse. Convolving that impulse image with a Gaussian of bandwidth `h` is equivalent to isotropic 2D KDE up to a constant scale factor (Silverman, 1986).
3. Threshold the density map, remove tiny connected components, and optionally skeletonise, prune, and dilate to recover a neurite-like mask.

For these data this matters: the earlier Frangi/Meijering/density attempts smoothed the **intensity field**, so every noisy pixel contributed to the response. Here only detected puncta contribute. Noise pixels that do not produce a valid scale-space response never enter the density estimate. The main failure mode is therefore missed puncta (LoG threshold too high) or false puncta (threshold too low), not direct propagation of pixel noise.

## References
- Silverman, *Density Estimation for Statistics and Data Analysis*. Chapman & Hall, 1986.
- Lindeberg, “Feature detection with automatic scale selection”, *International Journal of Computer Vision* 30(2):79-116, 1998.
- Levet et al., “SR-Tesseler: a method to segment and quantify localization-based super-resolution microscopy data”, *Nature Methods* 12(11):1065-1071, 2015.
- Andronov et al., “ClusterViSu, a method for clustering of protein complexes by Voronoi tessellation in super-resolution microscopy”, *Scientific Reports* 6:24084, 2016.

SR-Tesseler and ClusterViSu work in a directly analogous regime: noisy 2D point clouds of localisations. They use Voronoi densities rather than Gaussian KDE, but the key idea is the same: build density from a point process, not from raw intensity.


In [ ]:
import importlib.util
import itertools
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import convolve, gaussian_filter
from skimage.filters import threshold_otsu
from skimage.measure import label as cc_label, regionprops
from skimage.morphology import dilation, disk, skeletonize

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
for p in (SRC_ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)


def load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


_blobs = load_module(
    "synaptic_ssl_notebook_blobs",
    SRC_ROOT / "synaptic_ssl" / "pseudolabels" / "blobs.py",
)
_reassemble = load_module(
    "synaptic_ssl_notebook_reassemble",
    SRC_ROOT / "synaptic_ssl" / "utils_data" / "reassemble.py",
)

BlobPseudoCfg = _blobs.BlobPseudoCfg
detect_blobs_log = _blobs.detect_blobs_log
ImageCache = _reassemble.ImageCache
load_patch_records = _reassemble.load_patch_records

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 9


## Configuration

Keep the demo small: 8 patches, a 3x3 sweep over LoG threshold and KDE bandwidth, and two density-threshold options (`guarded_otsu` vs percentile). The goal is a fast visual sanity check, not full-dataset mask generation.


In [ ]:
PATCH_ROOT = REPO_ROOT / "data" / "patches_128"
EXCLUDE_PATTERNS = ["KONTROLA"]

STRUCTURAL_CHANNEL = 2
DEMO_INDEX_HINTS = (1, 3, 4, 7, 23, 50, 200, 400)

LOG_MIN_SIGMA = 0.7
LOG_MAX_SIGMA = 1.8
LOG_NUM_SIGMA = 5
LOG_OVERLAP = 0.5
LOG_EXCLUDE_BORDER = 5

LOG_THRESHOLDS = (0.003, 0.005, 0.01)
BANDWIDTHS = (3, 6, 10)
DENSITY_THRESHOLD_METHODS = (
    {"name": "guarded_otsu", "eta_min": 0.70, "fg_frac_max": 0.35},
    {"name": "percentile", "percentile": 98.0},
)

POSTPROCESS = {
    "min_cc_area": 60,
    "use_skeleton": True,
    "prune_len": 8,
    "dilate_radius": 2,
}

DISPLAY_CFG = {
    "log_threshold": 0.005,
    "bandwidth": 6,
    "thr_method": DENSITY_THRESHOLD_METHODS[0],
}

print("PATCH_ROOT:", PATCH_ROOT)
print("DEMO_INDEX_HINTS:", DEMO_INDEX_HINTS)
print(
    "GRID SIZE:",
    len(LOG_THRESHOLDS) * len(BANDWIDTHS) * len(DENSITY_THRESHOLD_METHODS),
)


## Load demo patches

The notebook uses `ImageCache` so each source image is reassembled at most once and the selected 128x128 patches are sliced from that cached full image. Demo indices are fixed to eight canonical patch positions and then resolved against the actual dataset length.


In [ ]:
def resolve_demo_positions(records, hints, n=8):
    if not records:
        return []

    resolved = []
    seen = set()
    max_idx = len(records) - 1
    for idx in hints:
        j = max(0, min(int(idx), max_idx))
        if j not in seen:
            resolved.append(j)
            seen.add(j)
        if len(resolved) == n:
            return resolved

    if len(records) <= n:
        return list(range(len(records)))

    extra = np.linspace(0, max_idx, num=n * 2, dtype=int)
    for j in extra:
        j = int(j)
        if j not in seen:
            resolved.append(j)
            seen.add(j)
        if len(resolved) == n:
            break
    return resolved[:n]


if not (PATCH_ROOT / "index.csv").exists():
    raise FileNotFoundError(
        f"PATCH_ROOT does not contain index.csv: {PATCH_ROOT}\n"
        "Update PATCH_ROOT to the extracted flat patch dataset before running the notebook."
    )

patch_records = load_patch_records(PATCH_ROOT, EXCLUDE_PATTERNS)
cache = ImageCache(PATCH_ROOT, patch_records)
DEMO_POSITIONS = resolve_demo_positions(patch_records, DEMO_INDEX_HINTS, n=8)

demo_patches = [cache.get_patch(pos) for pos in DEMO_POSITIONS]
demo_structurals = [
    patch[STRUCTURAL_CHANNEL].astype(np.float32, copy=False)
    for patch in demo_patches
]
demo_labels = [
    f"pos={pos} | img={patch_records[pos]['image_index']} | {patch_records[pos]['filename']}"
    for pos in DEMO_POSITIONS
]

print(f"Loaded {len(patch_records)} patch records from {PATCH_ROOT}")
print(f"Using {len(DEMO_POSITIONS)} demo patches:")
for label in demo_labels:
    print("  ", label)


## KDE helpers

`puncta_to_density` implements the centroid KDE approximation directly: rasterise LoG centroids into an impulse image with `np.add.at`, then blur with `gaussian_filter`. Thresholding can use either a high percentile or guarded Otsu, and the binary mask can optionally be skeletonised, branch-pruned, and dilated.


In [ ]:
_NEIGHBOUR_KERNEL = np.array(
    [[1, 1, 1], [1, 0, 1], [1, 1, 1]],
    dtype=np.uint8,
)


def make_log_cfg(log_threshold):
    return BlobPseudoCfg(
        structural_channel=STRUCTURAL_CHANNEL,
        log_min_sigma=LOG_MIN_SIGMA,
        log_max_sigma=LOG_MAX_SIGMA,
        log_num_sigma=LOG_NUM_SIGMA,
        log_threshold=float(log_threshold),
        log_overlap=LOG_OVERLAP,
        log_exclude_border=LOG_EXCLUDE_BORDER,
    )


def filter_min_area(mask, min_area):
    mask = np.asarray(mask, dtype=bool)
    if min_area <= 1 or not mask.any():
        return mask.copy()
    lbl = cc_label(mask, connectivity=2)
    out = np.zeros_like(mask, dtype=bool)
    for prop in regionprops(lbl):
        if prop.area >= int(min_area):
            out[lbl == prop.label] = True
    return out


def puncta_to_density(centroids, shape, bandwidth):
    impulse_image = np.zeros(shape, dtype=np.float32)
    if centroids.size == 0:
        return impulse_image, impulse_image.copy()

    rows = np.clip(np.rint(centroids[:, 0]).astype(int), 0, shape[0] - 1)
    cols = np.clip(np.rint(centroids[:, 1]).astype(int), 0, shape[1] - 1)
    np.add.at(impulse_image, (rows, cols), 1.0)

    density_map = gaussian_filter(
        impulse_image,
        sigma=float(bandwidth),
        mode="constant",
        cval=0.0,
    ).astype(np.float32)
    return impulse_image, density_map


def guarded_otsu_threshold(image, eta_min=0.70, fg_frac_max=0.35):
    nz = image[image > 0]
    if nz.size <= 1 or float(nz.max() - nz.min()) <= 0:
        return float("inf")

    thr = float(threshold_otsu(nz))
    fg = nz[nz > thr]
    bg = nz[nz <= thr]
    if fg.size == 0 or bg.size == 0:
        return float("inf")

    var_total = float(nz.var())
    if var_total <= 0.0:
        return float("inf")

    w_fg = fg.size / nz.size
    mean_all = float(nz.mean())
    var_between = (
        w_fg * (float(fg.mean()) - mean_all) ** 2
        + (1.0 - w_fg) * (float(bg.mean()) - mean_all) ** 2
    )
    eta = var_between / var_total
    if eta < float(eta_min):
        return float("inf")

    fg_frac = float((image > thr).mean())
    if fg_frac > float(fg_frac_max):
        return float("inf")
    return thr


def density_threshold(density_map, thr_method):
    name = thr_method["name"]
    if name == "guarded_otsu":
        return guarded_otsu_threshold(
            density_map,
            eta_min=thr_method.get("eta_min", 0.70),
            fg_frac_max=thr_method.get("fg_frac_max", 0.35),
        )
    if name == "percentile":
        nz = density_map[density_map > 0]
        if nz.size == 0:
            return float("inf")
        return float(np.percentile(nz, thr_method.get("percentile", 98.0)))
    raise ValueError(f"Unknown threshold method: {name}")


def skeleton_neighbour_count(skeleton):
    counts = convolve(
        skeleton.astype(np.uint8),
        _NEIGHBOUR_KERNEL,
        mode="constant",
        cval=0,
    )
    out = np.zeros_like(counts, dtype=np.int8)
    out[skeleton] = counts[skeleton]
    return out


def walk_branch(skeleton, neighbour_count, start, max_len):
    h, w = skeleton.shape
    path = [start]
    prev = None
    current = start

    while len(path) < int(max_len):
        r, c = current
        next_pixel = None
        n_forward = 0
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                rr = r + dr
                cc = c + dc
                if not (0 <= rr < h and 0 <= cc < w):
                    continue
                if not skeleton[rr, cc]:
                    continue
                if prev is not None and (rr, cc) == prev:
                    continue
                n_forward += 1
                next_pixel = (rr, cc)
        if next_pixel is None or n_forward != 1:
            break
        if neighbour_count[next_pixel] >= 3:
            break
        prev = current
        current = next_pixel
        path.append(current)
    return path


def prune_skeleton_branches(skeleton, max_len):
    skeleton = np.asarray(skeleton, dtype=bool)
    if max_len <= 0 or not skeleton.any():
        return skeleton.copy()

    work = skeleton.copy()
    while True:
        neighbour_count = skeleton_neighbour_count(work)
        endpoints = np.argwhere(neighbour_count == 1)
        if endpoints.size == 0:
            break

        removed_any = False
        for r, c in endpoints:
            if not work[r, c]:
                continue
            branch = walk_branch(work, neighbour_count, (int(r), int(c)), max_len)
            if len(branch) < int(max_len):
                for br, bc in branch:
                    work[br, bc] = False
                removed_any = True
        if not removed_any:
            break
    return work


def kde_dendrite_mask(
    structural,
    log_cfg,
    bandwidth,
    thr_method,
    *,
    min_cc_area=60,
    use_skeleton=True,
    prune_len=8,
    dilate_radius=2,
):
    blobs = detect_blobs_log(structural, log_cfg)
    impulses, density = puncta_to_density(blobs, structural.shape, bandwidth)
    threshold = density_threshold(density, thr_method)

    raw_mask = density > threshold
    raw_mask = filter_min_area(raw_mask, min_cc_area)

    skeleton = np.zeros_like(raw_mask, dtype=bool)
    pruned = np.zeros_like(raw_mask, dtype=bool)
    if use_skeleton and raw_mask.any():
        skeleton = skeletonize(raw_mask)
        pruned = prune_skeleton_branches(skeleton, prune_len)
        if dilate_radius > 0 and pruned.any():
            mask = dilation(pruned, disk(int(dilate_radius)))
        else:
            mask = pruned
    else:
        mask = raw_mask.copy()

    return {
        "blobs": blobs,
        "impulses": impulses,
        "density": density,
        "threshold": float(threshold),
        "raw_mask": raw_mask.astype(bool, copy=False),
        "skeleton": skeleton,
        "pruned_skeleton": pruned,
        "mask": mask.astype(bool, copy=False),
    }


def thr_method_label(thr_method):
    if thr_method["name"] == "guarded_otsu":
        return f"guarded_otsu(eta>={thr_method.get('eta_min', 0.70):.2f})"
    if thr_method["name"] == "percentile":
        return f"percentile_{thr_method.get('percentile', 98.0):.1f}"
    return thr_method["name"]


def summarise_patch_result(result):
    mask = result["mask"]
    lbl = cc_label(mask, connectivity=2)
    areas = [prop.area for prop in regionprops(lbl)]
    density_peak = float(np.max(result["density"])) if result["density"].size else 0.0
    return {
        "n_blobs": int(len(result["blobs"])),
        "fg_fraction": float(mask.mean()),
        "n_cc": int(lbl.max()),
        "largest_cc": int(max(areas, default=0)),
        "density_peak": density_peak,
    }


## Parameter sweep

Sweep `log_threshold in {0.003, 0.005, 0.01}`, `h in {3, 6, 10}`, and two density thresholds (`guarded_otsu` vs percentile). Without labels there is no exact objective, so the notebook reports simple patch-level diagnostics: number of LoG puncta, foreground fraction, number of connected components, and largest connected component size.


In [ ]:
sweep_rows = []
per_config_results = {}

for log_threshold, bandwidth, thr_method in itertools.product(
    LOG_THRESHOLDS,
    BANDWIDTHS,
    DENSITY_THRESHOLD_METHODS,
):
    log_cfg = make_log_cfg(log_threshold)
    patch_summaries = []
    patch_results = []
    for structural in demo_structurals:
        result = kde_dendrite_mask(
            structural,
            log_cfg,
            bandwidth,
            thr_method,
            **POSTPROCESS,
        )
        patch_results.append(result)
        patch_summaries.append(summarise_patch_result(result))

    row = {
        "log_threshold": float(log_threshold),
        "bandwidth": float(bandwidth),
        "thr_method": thr_method_label(thr_method),
        "mean_blobs": float(np.mean([s["n_blobs"] for s in patch_summaries])),
        "mean_fg_fraction": float(np.mean([s["fg_fraction"] for s in patch_summaries])),
        "mean_cc": float(np.mean([s["n_cc"] for s in patch_summaries])),
        "mean_largest_cc": float(np.mean([s["largest_cc"] for s in patch_summaries])),
    }
    key = (
        float(log_threshold),
        float(bandwidth),
        thr_method_label(thr_method),
    )
    per_config_results[key] = patch_results
    sweep_rows.append(row)


def recommendation_score(row, target_fg=0.10):
    return (
        abs(row["mean_fg_fraction"] - target_fg)
        + 0.015 * row["mean_cc"]
        - 0.002 * row["mean_largest_cc"]
        - 0.001 * row["mean_blobs"]
    )


sweep_rows = sorted(
    sweep_rows,
    key=lambda row: (row["log_threshold"], row["bandwidth"], row["thr_method"]),
)
auto_recommendation = min(sweep_rows, key=recommendation_score)

header = (
    f"{'log_thr':>7}  {'h':>4}  {'threshold':>24}  {'mean_blobs':>10}  "
    f"{'mean_fg%':>9}  {'mean_cc':>8}  {'mean_largest_cc':>15}"
)
print(header)
print("-" * len(header))
for row in sweep_rows:
    print(
        f"{row['log_threshold']:7.3f}  "
        f"{row['bandwidth']:4.0f}  "
        f"{row['thr_method']:>24}  "
        f"{row['mean_blobs']:10.2f}  "
        f"{100 * row['mean_fg_fraction']:8.2f}%  "
        f"{row['mean_cc']:8.2f}  "
        f"{row['mean_largest_cc']:15.2f}"
    )

print()
print(
    "Auto suggestion:",
    f"log_threshold={auto_recommendation['log_threshold']:.3f},",
    f"h={auto_recommendation['bandwidth']:.0f},",
    auto_recommendation["thr_method"],
)

display_key = (
    float(DISPLAY_CFG["log_threshold"]),
    float(DISPLAY_CFG["bandwidth"]),
    thr_method_label(DISPLAY_CFG["thr_method"]),
)
print("DISPLAY_CFG:", display_key)


## Visual sanity check on 8 patches

Rows are demo patches. Columns show the raw structural channel, LoG centroids, KDE density map, binary mask, and the mask overlaid on the raw structural image.


In [ ]:
display_key = (
    float(DISPLAY_CFG["log_threshold"]),
    float(DISPLAY_CFG["bandwidth"]),
    thr_method_label(DISPLAY_CFG["thr_method"]),
)
display_results = per_config_results[display_key]

fig, axes = plt.subplots(
    len(DEMO_POSITIONS),
    5,
    figsize=(16, 2.7 * len(DEMO_POSITIONS)),
    constrained_layout=True,
)
if len(DEMO_POSITIONS) == 1:
    axes = np.asarray([axes])

col_titles = [
    "structural raw",
    "LoG puncta overlay",
    "KDE density",
    "binary mask",
    "mask on raw",
]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title)

for row, (pos, label, structural, result) in enumerate(
    zip(DEMO_POSITIONS, demo_labels, demo_structurals, display_results)
):
    vmax = float(np.quantile(structural, 0.995)) if structural.size else 1.0
    vmax = max(vmax, 1e-6)
    metrics = summarise_patch_result(result)

    axes[row, 0].imshow(structural, cmap="gray", vmin=0.0, vmax=vmax)

    axes[row, 1].imshow(structural, cmap="gray", vmin=0.0, vmax=vmax)
    if len(result["blobs"]):
        blob_sigma = np.maximum(result["blobs"][:, 2], 0.5)
        axes[row, 1].scatter(
            result["blobs"][:, 1],
            result["blobs"][:, 0],
            s=(blob_sigma ** 2) * 20.0,
            facecolors="none",
            edgecolors="lime",
            linewidths=0.8,
        )

    axes[row, 2].imshow(result["density"], cmap="magma")
    axes[row, 2].text(
        0.02,
        0.98,
        f"peak={metrics['density_peak']:.3g}",
        transform=axes[row, 2].transAxes,
        va="top",
        ha="left",
        fontsize=7,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
    )

    axes[row, 3].imshow(result["mask"], cmap="gray", interpolation="nearest")

    axes[row, 4].imshow(structural, cmap="gray", vmin=0.0, vmax=vmax)
    axes[row, 4].imshow(
        np.ma.masked_where(~result["mask"], result["mask"]),
        cmap="autumn",
        alpha=0.45,
        interpolation="nearest",
    )

    image_index = patch_records[pos]["image_index"]
    axes[row, 0].set_ylabel(
        f"idx {pos}\nimg {image_index}\nblobs={metrics['n_blobs']}\nfg={100 * metrics['fg_fraction']:.1f}%",
        rotation=0,
        ha="right",
        va="center",
        labelpad=35,
    )
    axes[row, 4].text(
        0.02,
        0.98,
        f"thr={result['threshold']:.3g}\ncc={metrics['n_cc']}\nmaxCC={metrics['largest_cc']}",
        transform=axes[row, 4].transAxes,
        va="top",
        ha="left",
        fontsize=7,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
    )

    for col in range(5):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

fig.suptitle(
    "KDE on LoG punctum centroids\n"
    f"log_threshold={display_key[0]:.3f}, h={display_key[1]:.0f}px, {display_key[2]}, post={POSTPROCESS}",
    y=1.02,
    fontsize=12,
)
plt.show()


## Initial recommendation and failure modes

Start with **`log_threshold=0.005`**, **`h=6 px`**, and **guarded Otsu**. On 128 px patches that bandwidth is usually large enough to bridge puncta along a dendrite without immediately merging nearby processes. `h=3` tends to track puncta too literally and leaves broken branches; `h=10` is the first setting likely to over-merge neighbouring structures.

Main failure modes:
- **Sparse dendrite segments:** real branches break when LoG misses puncta or the density threshold is too strict.
- **LoG too permissive:** false puncta create diffuse bridges and inflate the mask.
- **LoG too strict:** density collapses to isolated islands.
- **Large bright clusters:** skeletonise → prune → dilate helps, but very round somatic or junction-like clusters can still thicken the output.

The key win over intensity KDE is unchanged: only detected puncta contribute to the density estimate, so background noise pixels cannot leak into the mask unless they first survive the LoG detector.
